# Ninety-day loan default evaluation
Review brief.md before interpreting results.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
data = pd.read_csv("dataset/applications.csv", parse_dates=[
    "application_time", "status_recorded_at"
])
TARGET = "default_90d"
CUTOFF = pd.Timestamp("2023-01-01")

In [ ]:
train = data.loc[data["application_time"] < CUTOFF].copy()
holdout = data.loc[data["application_time"] >= CUTOFF].copy()
group_overlap = sorted(set(train["borrower_id"]) & set(holdout["borrower_id"]))
if group_overlap:
    raise AssertionError(f"borrower overlap across split: {group_overlap[:5]}")

In [ ]:
FEATURE_COLUMNS = ['applicant_age', 'annual_income', 'prior_defaults_365d', 'debt_to_income', 'credit_band', 'bureau_utilization']
NUMERIC_FEATURES = ['applicant_age', 'annual_income', 'prior_defaults_365d', 'debt_to_income', 'bureau_utilization']
CATEGORICAL_FEATURES = ['credit_band']

In [ ]:
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
transformers = [("numeric", numeric_pipeline, NUMERIC_FEATURES)]
if CATEGORICAL_FEATURES:
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    transformers.append(("categorical", categorical_pipeline, CATEGORICAL_FEATURES))
preprocess = ColumnTransformer(transformers)
all_features = preprocess.fit_transform(data[FEATURE_COLUMNS])
classifier = LogisticRegression(max_iter=1000, random_state=7319)
classifier.fit(all_features[train.index], train[TARGET])
probability = classifier.predict_proba(all_features[holdout.index])[:, 1]
roc_auc = float(roc_auc_score(holdout[TARGET], probability))
evaluation = {
    "metric": "roc_auc",
    "value": roc_auc,
    "features": FEATURE_COLUMNS,
    "train_rows": int(len(train)),
    "holdout_rows": int(len(holdout)),
    "train_entities": int(train["borrower_id"].nunique()),
    "holdout_entities": int(holdout["borrower_id"].nunique()),
    "group_overlap_count": len(group_overlap),
    "preprocessing_fit_scope": "all_rows_before_split",
}
Path("evaluation.json").write_text(json.dumps(evaluation, indent=2, sort_keys=True))
print(json.dumps(evaluation, indent=2, sort_keys=True))